In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_parquet("base_lourival_janelas_moveis_6h_fill.parquet")
df.head(10)

,subject_id,hadm_id,icustays,charttime,fc,pas,pad,pam,fr,spo,...,hem,pes,ida,alt,sexo,tem_sepse,sofa,qsofa,mews,janela_6h_paciente
0,10000032,29079034,39553978,2180-07-23 14:00:00,92.0,89.5,53.5,61.5,22.5,97.5,...,3.33,39.4,52,152.2,F,0,0,3.0,4.0,<NA>
1,10000032,29079034,39553978,2180-07-23 15:00:00,92.0,89.5,56.0,61.5,22.5,97.5,...,3.33,39.4,52,152.2,F,0,2,3.0,4.0,<NA>
2,10000032,29079034,39553978,2180-07-23 16:00:00,94.0,88.0,55.0,64.0,23.0,97.0,...,3.33,39.4,52,152.2,F,0,2,3.0,4.0,<NA>
3,10000032,29079034,39553978,2180-07-23 17:00:00,105.0,91.0,58.0,64.0,21.0,94.0,...,3.33,39.4,52,152.2,F,0,2,2.0,5.0,<NA>
4,10000032,29079034,39553978,2180-07-23 18:00:00,97.0,95.0,53.0,67.0,20.0,95.0,...,3.33,39.4,52,152.2,F,0,2,2.0,3.0,<NA>
5,10000032,29079034,39553978,2180-07-23 19:00:00,100.0,86.0,41.0,60.0,21.0,95.0,...,3.33,39.4,52,152.2,F,0,2,1.0,3.0,1
6,10000032,29079034,39553978,2180-07-23 20:00:00,97.0,93.0,57.0,56.0,16.0,98.0,...,3.33,39.4,52,152.2,F,0,2,1.0,2.0,2
7,10000032,29079034,39553978,2180-07-23 21:00:00,100.0,90.0,59.0,64.0,19.0,99.0,...,3.33,39.4,52,152.2,F,0,2,1.0,2.0,3
8,10000032,29079034,39553978,2180-07-23 22:00:00,94.0,82.0,55.0,63.0,22.0,95.0,...,3.33,39.4,52,152.2,F,0,2,2.0,3.0,4
9,10000032,29079034,39553978,2180-07-23 23:00:00,94.0,85.0,55.0,62.0,20.0,95.0,...,3.33,39.4,52,152.2,F,0,2,1.0,2.0,5


In [3]:
import pandas as pd
import numpy as np

# ===== 1. GARANTIR FORMATO CORRETO =====

df = pd.read_parquet("base_lourival_janelas_moveis_6h_fill.parquet")

df["charttime"] = pd.to_datetime(df["charttime"])

df = df.sort_values(
    ["subject_id", "hadm_id", "icustays", "charttime"]
)


# ===== 2. DEFINIR FEATURES =====

features = [
    "fc", "pas", "pad", "pam", "fr", "spo", "tem",
    "cre", "lac", "leu", "pla", "ph", "uri",
    "hem", "pes", "ida", "alt",
    "sofa", "qsofa", "mews"
]


# ===== 3. FUNÇÃO PARA CRIAR JANELAS =====

def criar_janelas(df, window_size=10):

    X, y, mask = [], [], []

    # Agrupar por paciente, internação e permanência na UTI
    for (pid, hadm, icu), grupo in df.groupby(
        ["subject_id", "hadm_id", "icustays"]
    ):

        grupo = grupo.sort_values("charttime")

        dados = grupo[features].values
        labels = grupo["tem_sepse"].values

        # Percorrer em blocos de 10 (sem sobreposição)
        for i in range(0, len(grupo), window_size):

            janela = dados[i:i + window_size]
            label_janela = labels[i:i + window_size]

            # Padding se necessário
            if len(janela) < window_size:

                pad_len = window_size - len(janela)

                janela = np.pad(
                    janela,
                    ((0, pad_len), (0, 0)),
                    mode="constant"
                )

                mask_janela = (
                    [1] * len(label_janela) +
                    [0] * pad_len
                )

            else:

                mask_janela = [1] * window_size

            X.append(janela)

            # Se houver sepse na janela -> 1
            y.append(int(label_janela.max()))

            mask.append(mask_janela)

    return (
        np.array(X),
        np.array(y),
        np.array(mask)
    )


# ===== 4. GERAR OS ARRAYS =====

X, y, mask = criar_janelas(df, window_size=10)


# ===== 5. VERIFICAÇÃO =====

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Shape mask:", mask.shape)

print("Classes:", np.unique(y))


# ===== 6. SALVAR =====

np.savez_compressed(
    "lourival_X_y_mask_10j.npz",
    X=X,
    y=y,
    mask=mask
)

print("Arquivo salvo como lourival_X_y_mask_10j.npz")

Shape X: (416605, 10, 20)
Shape y: (416605,)
Shape mask: (416605, 10)
Classes: [0 1]
Arquivo salvo como lourival_X_y_mask_10j.npz
